In [3]:
from PIL import Image
import os
import json
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
import json
import os
import numpy as np
from PIL import Image
from sklearn.preprocessing import MultiLabelBinarizer

with open('classes.json', 'r') as f:
    class_data = json.load(f)
    class_list = class_data['classes'] 

print(f"Classes identified: {class_list}")

X = []
y = []

for filename in os.listdir('images'):
    if filename.endswith('.jpg'):
        img_path = os.path.join('images', filename)
        label_path = os.path.join('labels', filename.replace('.jpg', '.json'))
        
        if os.path.exists(label_path):
            # Process Image
            with Image.open(img_path) as img:
                img = img.resize((64, 64)).convert('L')
                X.append(np.array(img).flatten())
            
            # Process Label
            with open(label_path, 'r') as f:
                label_data = json.load(f)
                # Double check the key inside the INDIVIDUAL label files
                # (e.g., arduino_kit_01.json)
                y.append(label_data.get('items_present', []))

X = np.array(X)
mlb = MultiLabelBinarizer(classes=class_list)
y_bin = mlb.fit_transform(y)


Classes identified: ['Arduino Kit', 'Raspberry Pi', 'STM32 Microcontroller', 'USB Cable', 'HDMI Cable', 'Soldering Station']


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [21]:
rf = RandomForestClassifier()
X_train,X_test,y_train,y_test = train_test_split(X,y_bin,test_size=0.2, random_state=42)

rf.fit(X_train,y_train)
predictions = rf.predict(X_test)
probabilities = rf.predict_proba(X_test)

In [ ]:
import json

final_output = []

# 1. Get probabilities
probs = rf.predict_proba(X_test)
class_names = mlb.classes_

for i in range(len(X_test)):
    scene_results = []
    
    for class_index, class_name in enumerate(class_names):
        class_prob_array = probs[class_index][i]
        
        if len(class_prob_array) == 2:
            confidence = class_prob_array[1]
        else:
            confidence = 0.0
            
        if confidence > 0.5:
            scene_results.append({
                "name": class_name,
                "confidence": round(float(confidence), 2)
            })
    
    entry = {
        "scene_id": f"shelf_{i:02d}", 
        "predictions": scene_results
    }
    final_output.append(entry)

with open('predictions.json', 'w') as f:
    json.dump(final_output, f, indent=4)
    
print("Saved predictions.json!")

In [ ]:
import json
from datetime import datetime

CONF_THRESHOLD = 0.90 

inventory_db = {
    "Arduino Kit": 5,
    "Raspberry Pi": 2,
    "STM32 Microcontroller": 0,
    "USB Cable": 15,
    "HDMI Cable": 0,
    "Soldering Station": 1
}

try:
    with open('predictions.json', 'r') as f:
        predictions_data = json.load(f)
except FileNotFoundError:
    print("Please run your prediction cell first to generate predictions.json!")
    predictions_data = []

audit_events = []
total_accepted = 0
total_uncertain = 0

for scene in predictions_data:
    scene_id = scene['scene_id']
    accepted_predictions = []
    
    for pred in scene['predictions']:
        item_name = pred['name']
        confidence = pred['confidence']

        # Check if the confidence meets our policy threshold
        if confidence >= CONF_THRESHOLD:
            accepted_predictions.append(pred)
            total_accepted += 1
        else:
            total_uncertain += 1
            audit_events.append({
                "timestamp": datetime.utcnow().isoformat(),
                "scene_id": scene_id,
                "item": item_name,
                "event_type": "UNCERTAIN",
                "confidence": confidence,
                "recommended_action": "Manual review required"
            })


    for pred in accepted_predictions:
        item_name = pred['name']
        confidence = pred['confidence']
        
        db_quantity = inventory_db.get(item_name, 0)
        
        if db_quantity > 0:

            audit_events.append({
                "timestamp": datetime.utcnow().isoformat(),
                "scene_id": scene_id,
                "item": item_name,
                "event_type": "VERIFIED",
                "confidence": confidence,
                "recommended_action": "None"
            })
        else:

            audit_events.append({
                "timestamp": datetime.utcnow().isoformat(),
                "scene_id": scene_id,
                "item": item_name,
                "event_type": "DISCREPANCY",
                "confidence": confidence,
                "recommended_action": "Update inventory database"
            })

print(f"Accepted Predictions: {total_accepted}")
print(f"Uncertain Predictions: {total_uncertain}")

if audit_events:
    with open('audit_log.jsonl', 'w') as f:
        for event in audit_events:
 
            f.write(json.dumps(event) + '\n')
            
    print("\nSuccessfully wrote audit events to audit_log.jsonl")

Please run your prediction cell first to generate predictions.json!
--- Policy Filtering Results ---
Accepted Predictions: 0
Uncertain Predictions: 0
